# Reversed ML classifier

## Found out to be not super doable because as for inner-bin spectrum we do not have MSMS data. 

## Approach: 

### Step 1: Data Preparation
Gather Data:

Collect raw spectra data from samples.
Organize spectra into bins based on similarity (clustering or predefined rules).
Retrieve reference spectra from the known library.

Preprocessing:

Normalize spectra data for consistency.
Apply dimensionality reduction techniques (e.g., PCA, t-SNE) if needed.
Generate features (e.g., peak intensities, m/z ratios) for classification.


Dataset Creation:

Label each bin and reference spectrum.
Create a structured dataset where bins act as classes and spectra within bins serve as training examples.


### Step 2: Model Selection
Baseline Classifier:

Use a traditional machine learning classifier from scikit-learn such as:
Random Forest
Support Vector Machine (SVM)
k-Nearest Neighbors (kNN)
Logistic Regression
Deep Learning Approach (Optional):

Implement a deep feedforward neural network (DNN) using TensorFlow/PyTorch if more complex relationships exist.
Use batch normalization, dropout, and activation functions (ReLU, softmax) for better generalization.
### Step 3: Training the Classifier
Training Strategy:

Either:
Perform 10-fold cross-validation to ensure robustness.
Use a 70/30 train-test split within each sample.
Loss Function & Metrics:

Use cross-entropy loss for multi-class classification.
Evaluate accuracy, F1-score, and AUC-ROC for model assessment.
### Step 4: Testing & Heatmap Generation
Classify Known Library Compounds:

Each compound in the library is classified into the most probable bin.
Build Probability Heatmap:

Rows = Known compounds.
Columns = Bins.
Cell values = Classification probabilities.
Use seaborn.heatmap() for visualization.
Sanity Check:

Verify that each bin has 1-2 compounds with high probability.
Step 5: Optimization & Finalization
Hyperparameter Tuning:

Use GridSearchCV or Optuna for optimal classifier parameters.
Tune neural network architecture (layers, dropout, learning rate).
Model Evaluation:

Test model on unseen sample bins to verify generalization.
Deployment:

Save the trained model using joblib (for scikit-learn) or torch.save() (for PyTorch).
Wrap in a simple pipeline for practical use.


In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read the data and skip the first row
df = pd.read_csv('/Users/ellayoung/Desktop/metabolo_confi_score/data/base_data_202501161036.txt', sep='|', skipinitialspace=True)
df = df.iloc[1:]  # Skip the first row

# Clean column names and data
df.columns = df.columns.str.strip()
for col in df.columns:
    df[col] = df[col].str.strip() if df[col].dtype == 'object' else df[col]

def clean_curly_list(s):
    try:
        values = s.strip('{}').split(',')
        return [float(x.strip()) for x in values if x.strip()]
    except Exception as e:
        print(f"Error processing string: {s}")
        print(f"Error: {e}")
        return []

# Get column names
ri_col = [col for col in df.columns if 'ri_list' in col][0]
mass_col = [col for col in df.columns if 'mass_list' in col][0]

df = df.drop(columns=['Unnamed: 0', 'Unnamed: 7'])


In [20]:
df

,compound_splash,splash_list,sample_list,mass_list,ri_list,rt_list
1,splash10-0002-0900000000-8b8db3aac8e64d992444,{splash10-0002-0900000000-0002ac8905a75db691c0...,"{BioRec001_MX677489_negBA_preJepsen001,BioRec0...","[268.134473546, 268.1344888311, 268.1344893744...","[85.805916671, 85.8565840456, 85.9652353215, 8...","[85.1274185181, 85.1911621094, 85.2319793701, ..."
2,splash10-0002-0900000000-e9169f326437832af5f0,{splash10-0002-0900000000-1d466a5e8896671b47df...,"{BioRec001_MX677489_negBA_preJepsen001,BioRec0...","[149.0642914092, 149.0642916077, 149.064304215...","[126.2204520736, 126.6012951637, 126.655822610...","[126.9425735474, 126.9540710449, 126.980484008..."
3,splash10-0006-0900000000-32ab6d568fb5e2303927,{splash10-0002-0900000000-6770cc1a51ebc6463a0e...,"{BioRec001_MX749154_negBA_postSchoeman0010,Bio...","[140.1478138343, 140.1481128222, 140.148185505...","[97.8899736436, 98.2200516708, 98.227495178, 9...","[98.6423568726, 99.102432251, 99.127532959, 99..."
4,splash10-0006-9000000000-d8736691c0b20b33728b,{splash10-0006-1900000000-7d41b654e8a0315515f5...,{BioRec02_MX612252_NegBA_postLongevity_Female0...,"[92.0628704714, 92.0629049885, 92.0629187112, ...","[116.5874311094, 117.0185031282, 117.031938193...","[118.0083770752, 118.041809082, 118.0741958618..."
5,splash10-0006-9100000000-b60c6c104cfc41a9c1e9,{splash10-0002-1200900000-34567ab07246f41ec39f...,"{BioRec001_MX749154_negBA_postSchoeman0010,Bio...","[137.0433259995, 137.0433554715, 137.043371296...","[55.0350771929, 55.1980512128, 55.2308033389, ...","[56.5179405212, 56.7735977173, 56.7876663208, ..."
6,splash10-000i-0900000000-41cfd52ada5876c57771,{splash10-0006-0922000000-4272ac6a952c86da1524...,"{BioRec001_MX677489_negBA_preJepsen001,BioRec0...","[135.0486129289, 135.0486361563, 135.048645601...","[132.1225098028, 132.1334245467, 132.149320817...","[133.6080932617, 133.6629943848, 133.693344116..."
7,splash10-000i-0900000000-85e7cc6b26b4d5cbec4e,{splash10-0006-0229000000-343114e71ad95a6b118b...,"{BioRec001_MX677489_negBA_preJepsen001,BioRec0...","[180.1479841137, 180.147984513, 180.1479909792...","[136.6093932871, 136.6846574994, 136.713912592...","[135.920211792, 136.3144836426, 136.629486084,..."
8,splash10-001i-0090000000-2b8e455027295574c51c,{splash10-0002-0940000000-7bd15a5b6f4d7f9bbf28...,"{BioRec001_MX806650_negBA_postRima0010,BioRec0...","[234.1277901445, 234.128639229, 234.129037492,...","[11.6401227953, 11.7078299943, 11.7264476239, ...","[11.1637954712, 11.4586305618, 11.535036087, 1..."
9,splash10-001i-0900000000-f7b21c968de3ebbd77a1,{splash10-000f-9300000000-769895f38ae984f28b42...,"{MtdBlank001_MX749154_negBA_preSchoeman0001,Mt...","[133.0950799524, 133.0950853598, 133.095097948...","[143.2031215132, 143.2104113848, 143.297030539...","[143.0800476074, 143.1083679199, 143.148468017..."
10,splash10-004i-0009000000-54095bb57be75203a8f2,{splash10-0002-0900000000-b25da4fd51d81d52d68a...,"{BioRec001_MX677489_negBA_preJepsen001,BioRec0...","[378.1995888568, 378.2015121681, 378.201598212...","[97.2038469894, 97.2522576617, 97.3258824153, ...","[97.7923355103, 97.8739242554, 97.9055328369, ..."


In [17]:
# Define a function to convert a string like "{val1,val2,...}" into a list of floats
def convert_to_list(value):
    try:
        # Remove curly braces and split by comma, then convert each item to float
        return list(map(float, value.strip('{}').split(',')))
    except Exception as e:
        print(f"Error converting value: {value} - {e}")
        return []

# Apply conversion to each relevant column
df['mass_list'] = df['mass_list'].apply(convert_to_list)
df['ri_list']   = df['ri_list'].apply(convert_to_list)
df['rt_list']   = df['rt_list'].apply(convert_to_list)


In [19]:
import pandas as pd

records = []

# Iterate over each bin (each row in the DataFrame)
for idx, row in df.iterrows():
    bin_label = row['compound_splash']
    
    # Determine the number of entries to iterate over using the minimum length
    n_entries = min(len(row['mass_list']), len(row['ri_list']), len(row['rt_list']))
    
    # Optionally, log a warning if the lengths are not the same
    if not (len(row['mass_list']) == len(row['ri_list']) == len(row['rt_list'])):
        print(f"Warning: Row {idx} has mismatched list lengths: "
              f"mass_list({len(row['mass_list'])}), "
              f"ri_list({len(row['ri_list'])}), "
              f"rt_list({len(row['rt_list'])}). Using {n_entries} entries.")
    
    for i in range(n_entries):
        records.append({
            'mass': row['mass_list'][i],
            'ri': row['ri_list'][i],
            'rt': row['rt_list'][i],
            'bin': bin_label  # Label for this measurement
        })

# Convert the list of records into a DataFrame.
new_df = pd.DataFrame(records)

# Check the new DataFrame
print(new_df.shape)
print(new_df.head())


(538774, 4)
         mass         ri         rt  \
0  268.134474  85.805917  85.127419   
1  268.134489  85.856584  85.191162   
2  268.134489  85.965235  85.231979   
3  268.134530  86.105371  85.241608   
4  268.134562  86.154969  85.260880   

                                             bin  
0  splash10-0002-0900000000-8b8db3aac8e64d992444  
1  splash10-0002-0900000000-8b8db3aac8e64d992444  
2  splash10-0002-0900000000-8b8db3aac8e64d992444  
3  splash10-0002-0900000000-8b8db3aac8e64d992444  
4  splash10-0002-0900000000-8b8db3aac8e64d992444  
